# LSTM — Hyperparameter Optimisation

Based on LSTM_dave_adaption1. Key differences:
- **No heavy preprocessing** — features are imported already lagged, rolled and cleaned
- **Minimal preprocessing** — cyclical time features + StandardScaler only
- **Random search** hyperparameter optimisation on training set before CV
- Best params flow into 12-fold CV and final model automatically

## Structure
| Section | Description |
|---|---|
| Imports | Libraries incl. StandardScaler |
| Load data | Import pre-processed feature file |
| Minimal preprocessing | Cyclical features + NaN check |
| Define X / y | Feature matrix + target |
| Train/test split | 80/20 time-ordered |
| Sequence builder | Sliding-window helper |
| Random search | Tune LSTM hyperparameters on train set only |
| 12-fold CV | LSTM vs naive with best params |
| Summary table | MAE / RMSE / MASE per month |
| Monthly plots | Best + worst week per fold |
| Final model | Train on 80% with best params |

In [ ]:
import sys
sys.path.append('../../Data_Processing')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

import optuna
from optuna.visualization import matplotlib as optuna_mpl

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

from keras import models, Input, layers, optimizers
from keras.callbacks import EarlyStopping
from numpy.lib.stride_tricks import sliding_window_view

from full_cleaning_preprocessing_script import run_full_preprocessing
from xgboost_feature_selection_script import prepare_xgboost_features, build_xgboost_dataset

## Load Data

In [ ]:
TARGET    = 'el_price_dol_MWh_OTA2201'
NAIVE_COL = 'el_price_dol_MWh_OTA2201__lag_24h'   # log-scale — back-transform with np.exp()

df = run_full_preprocessing(
    data_folder = '/Users/Dave/code/DaveGiGi/98-Project-Duckstradamus/Data_Processing/data_input',
    start_date  = '2019-01-01',
    end_date    = '2024-12-31',
)

df = prepare_xgboost_features(
    df                  = df,
    target              = TARGET,
    skew_threshold      = 0.9,
    target_corr_cutoff  = 0.20,
    top_n_lags          = 3,
    multicoll_threshold = 0.9,
    verbose             = True,
)

df = build_xgboost_dataset(df)

# Verify naive baseline column exists and check shape
print('Shape      :', df.shape)
print(f'Date range : {df.index.min().date()} → {df.index.max().date()}')
print('OTA2201 cols:', [c for c in df.columns if 'OTA2201' in c])
df.head()

## Window Configuration

Set `DATE_START` / `DATE_END` to run on a shorter slice (e.g. for a fast optimisation trial).
Set both to `None` to use the full dataset.

Reduce `CV_FOLDS` when the window is short — rule of thumb: `CV_FOLDS × 720 ≤ 60% of total rows`.

In [ ]:
DATE_START = None   # e.g. '2022-01-01' — None = use full dataset
DATE_END   = None   # e.g. '2024-12-31' — None = use full dataset
CV_FOLDS   = 12     # reduce to 6 for short windows (rule: CV_FOLDS × 720 ≤ 60% of rows)

# Fixed validation windows for the hyperparameter search
# Each tuple = (start, end) — must fall inside the 80% train split
SEARCH_WINDOWS = [
    ('2022-01-01', '2022-04-01'),   # NZ summer  (3 months)
    ('2022-04-01', '2022-07-01'),   # NZ autumn  (3 months)
    ('2022-07-01', '2022-10-01'),   # NZ winter  (3 months)
]

# datetime_utc12 is now the index — filter using df.index
if DATE_START:
    df = df[df.index >= DATE_START]
if DATE_END:
    df = df[df.index <= DATE_END]
# Do NOT reset_index — keep datetime index throughout

print(f"Active window : {df.index.min().date()} → {df.index.max().date()}")
print(f"Rows          : {len(df)}")
print(f"CV folds      : {CV_FOLDS}  (test_size=720 each → {CV_FOLDS * 720} rows as test)")
print(f"Search windows: {len(SEARCH_WINDOWS)} × 3-month fixed folds (NZ summer / autumn / winter 2022)")

## Minimal Preprocessing

Lagging, rolling, and ffill are already done upstream.
Steps here:
1. Sort by datetime
2. Add cyclical time features (hour, day-of-week, day-of-year)
3. NaN check

StandardScaler is applied **per fold** in the CV loop (fitted on train only — no leakage).

In [ ]:
# Pipeline (run_full_preprocessing + prepare_xgboost_features) already handled:
# lags, rolling means, ffill, dropna, feature selection.
# This cell only adds time features and runs a safety NaN check.

df = df.sort_index()

# Cyclical time features — extracted from the datetime index
hour = df.index.hour
dow  = df.index.dayofweek
doy  = df.index.dayofyear

df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
df['dow_sin']  = np.sin(2 * np.pi * dow  / 7)
df['dow_cos']  = np.cos(2 * np.pi * dow  / 7)
df['doy_sin']  = np.sin(2 * np.pi * doy  / 365)
df['doy_cos']  = np.cos(2 * np.pi * doy  / 365)

# is_weekend — binary flag
df['is_weekend'] = pd.Series(dow, index=df.index).isin([5, 6]).astype(int)

# Safety NaN check — catches any pipeline edge cases
missing = df.isna().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print('NaNs found — dropping affected rows:')
    print(missing)
    df = df.dropna()
else:
    print('No NaNs — data is clean.')

print(f'\nFinal shape : {df.shape}')
print(f'Date range  : {df.index.min().date()} → {df.index.max().date()}')

## Define X and y

In [ ]:
# datetime_utc12 is now the index — only exclude the target column itself
exclude_cols = [TARGET]
feature_cols = [c for c in df.columns if c not in exclude_cols]

X_df = df[feature_cols]
y_s  = df[TARGET]   # log-scale prices — back-transform with np.exp() before metrics/plots

print('Feature matrix shape:', X_df.shape)
print('Number of features:  ', len(feature_cols))
print(f'Naive col present:   {NAIVE_COL in feature_cols}  ({NAIVE_COL})')

## Train / Test Split — 80 / 20 time-ordered

The 80% training slice is used for the hyperparameter search.
The 20% test slice is held out and only touched by the final model.

In [ ]:
split_index = int(len(df) * 0.8)

X_train_df = X_df.iloc[:split_index]
X_test_df  = X_df.iloc[split_index:]
y_train_s  = y_s.iloc[:split_index]
y_test_s   = y_s.iloc[split_index:]

print(f"Train: {X_train_df.shape}  {df.index[0].date()} → {df.index[split_index-1].date()}")
print(f"Test : {X_test_df.shape}   {df.index[split_index].date()} → {df.index[-1].date()}")

## Sequence Configuration

In [ ]:
INPUT_LENGTH  = 336   # 2 weeks of hourly history as input window
OUTPUT_LENGTH = 24    # predict the next 24 hours
HORIZON       = 1     # gap between last input row and first predicted row
STRIDE        = 24    # step between consecutive sequences (1 day)

## Sequence Builder

Converts a 2-D feature array + target array into (X_seq, y_seq) ready for the LSTM.
StandardScaler is fitted **outside** this function so it can be reused across splits.

In [ ]:
def build_sequences(X_arr, y_arr, input_length, output_length, horizon, stride, shuffle=False):
    """Return (X_seq, y_seq) sliding windows from pre-scaled arrays."""
    n = len(X_arr)
    last_start = n - input_length - horizon - output_length + 2
    starts = np.arange(0, last_start, stride)

    X_win = sliding_window_view(X_arr, input_length, axis=0)   # (n-IL+1, F, IL)
    X_seq = X_win[starts].transpose(0, 2, 1).astype(np.float32) # (S, IL, F)

    y_win = sliding_window_view(y_arr, output_length)           # (n-OL+1, OL)
    y_seq = y_win[starts + input_length + horizon - 1].astype(np.float32)

    if shuffle:
        idx = np.random.permutation(len(X_seq))
        X_seq, y_seq = X_seq[idx], y_seq[idx]

    return X_seq, y_seq

## Model Builder

Accepts a hyperparameter dict so the random search can swap params cleanly.

In [ ]:
def build_model(n_features, params):
    model = models.Sequential([
        Input(shape=(INPUT_LENGTH, n_features)),
        layers.LSTM(params['lstm1_units'],
                    activation='tanh',
                    return_sequences=True,
                    dropout=params['dropout']),
        layers.LSTM(params['lstm2_units'],
                    activation='tanh',
                    return_sequences=False,
                    dropout=params['dropout']),
        layers.Dense(params['dense_units'], activation='relu'),
        layers.Dropout(params['dropout']),
        layers.Dense(OUTPUT_LENGTH, activation='linear'),
    ])
    model.compile(
        loss='mae',       # MAE loss — robust to electricity price spikes
        optimizer=optimizers.Adam(learning_rate=params['learning_rate']),
        metrics=['mae'],
    )
    return model

## Optuna Hyperparameter Search\n\nUses **Optuna with TPE sampler** — learns from previous trials to suggest smarter candidates\nrather than sampling randomly. Pruning cuts bad trials early after the first fold.\n\nValidation uses the three fixed `SEARCH_WINDOWS` defined above (NZ summer / autumn / winter 2022),\ngiving consistent, season-aware coverage regardless of dataset size.\n\nParameters tuned:\n| Parameter | Options | Notes |\n|---|---|---|\n| `lstm1_units` | 32, 64, 128, 256 | Capacity of first recurrent layer |\n| `lstm2_units` | 16, 32, 64 | Capacity of second recurrent layer |\n| `dropout` | 0.1, 0.2, 0.3, 0.4 | Regularisation strength |\n| `dense_units` | 16, 32, 64 | Head capacity before output |\n| `learning_rate` | 0.001, 0.005, 0.01 | Adam step size |\n| `batch_size` | 16, 32, 64 | Sequences per gradient update |

In [ ]:
N_TRIALS = 20   # Optuna trials — fewer needed than random search due to TPE learning

# ── Build search folds from fixed SEARCH_WINDOWS ──────────────────────────
# datetime is now the index — use df.index directly
train_ts = df.index[:split_index]   # DatetimeIndex of training rows

search_folds = []
for start_date, end_date in SEARCH_WINDOWS:
    val_mask = (train_ts >= start_date) & (train_ts < end_date)
    tr_mask  = train_ts < start_date
    tr_idx   = np.where(tr_mask)[0]
    val_idx  = np.where(val_mask)[0]
    if len(tr_idx) > INPUT_LENGTH and len(val_idx) > INPUT_LENGTH + OUTPUT_LENGTH:
        search_folds.append((tr_idx, val_idx))
    else:
        print(f"  ⚠ Window {start_date}→{end_date} skipped — insufficient rows")

print(f"Search folds built: {len(search_folds)}")
for i, (tr, va) in enumerate(search_folds):
    print(f"  Fold {i+1}: train {len(tr)} rows  |  "
          f"val {train_ts[va[0]].date()} → {train_ts[va[-1]].date()}  ({len(va)} rows)")

X_train_arr = X_train_df.values
y_train_arr = y_train_s.values
n_features  = X_train_arr.shape[1]

# ── Optuna objective function ──────────────────────────────────────────────
def objective(trial):
    params = {
        'lstm1_units':   trial.suggest_categorical('lstm1_units',   [32, 64, 128, 256]),
        'lstm2_units':   trial.suggest_categorical('lstm2_units',   [16, 32, 64]),
        'dropout':       trial.suggest_categorical('dropout',       [0.1, 0.2, 0.3, 0.4]),
        'dense_units':   trial.suggest_categorical('dense_units',   [16, 32, 64]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.001, 0.005, 0.01]),
        'batch_size':    trial.suggest_categorical('batch_size',    [16, 32, 64]),
    }

    fold_maes = []
    for fold_i, (tr_idx, val_idx) in enumerate(search_folds):
        scaler   = StandardScaler()
        X_tr_sc  = scaler.fit_transform(X_train_arr[tr_idx])
        X_val_sc = scaler.transform(X_train_arr[val_idx])
        y_tr     = y_train_arr[tr_idx]
        y_val    = y_train_arr[val_idx]

        X_tr_seq,  y_tr_seq  = build_sequences(X_tr_sc,  y_tr,  INPUT_LENGTH, OUTPUT_LENGTH, HORIZON, STRIDE, shuffle=True)
        X_val_seq, y_val_seq = build_sequences(X_val_sc, y_val, INPUT_LENGTH, OUTPUT_LENGTH, HORIZON, STRIDE, shuffle=False)

        if len(X_tr_seq) == 0 or len(X_val_seq) == 0:
            continue

        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        model = build_model(n_features, params)
        model.fit(
            X_tr_seq, y_tr_seq,
            validation_data=(X_val_seq, y_val_seq),
            batch_size=params['batch_size'],
            epochs=50,
            callbacks=[es],
            verbose=0,
        )

        # Back-transform to NZD/MWh before scoring
        y_val_pred = np.exp(model.predict(X_val_seq, verbose=0).flatten())
        y_val_true = np.exp(y_val_seq.flatten())
        fold_mae   = mean_absolute_error(y_val_true, y_val_pred)
        fold_maes.append(fold_mae)

        # Report to Optuna — allows pruning after each fold
        trial.report(float(np.mean(fold_maes)), step=fold_i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return float(np.mean(fold_maes)) if fold_maes else float('inf')

# ── Run study ──────────────────────────────────────────────────────────────
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

BEST_PARAMS = study.best_params

print(f"\nBest trial    : #{study.best_trial.number}")
print(f"Best CV MAE   : {study.best_value:.4f} NZD/MWh")
print("\nBest parameters:")
for k, v in sorted(BEST_PARAMS.items()):
    print(f"  {k:<18}: {v}")

## Optuna Search Visualisation

In [ ]:
# ── Optimization history — MAE per trial, best so far highlighted ──────────
ax = optuna_mpl.plot_optimization_history(study)
ax.set_title('Optuna — Optimization History (CV MAE per trial)')
ax.set_xlabel('Trial number')
ax.set_ylabel('CV MAE')
plt.tight_layout()
plt.show()

# ── Parameter importances — which hyperparameters mattered most ────────────
ax = optuna_mpl.plot_param_importances(study)
ax.set_title('Optuna — Hyperparameter Importances')
plt.tight_layout()
plt.show()

# ── Trial summary table ────────────────────────────────────────────────────
trials_df = study.trials_dataframe().sort_values('value')
param_cols = [c for c in trials_df.columns if c.startswith('params_')]
print(trials_df[['number', 'value', 'state'] + param_cols]
      .rename(columns={'number': 'trial', 'value': 'cv_mae'})
      .to_string(index=False))

## 12-Fold Time-Series CV

Uses best params from the random search.
`test_size=720` ≈ 1 month of hourly data. Number of folds is controlled by `CV_FOLDS` in the Window Configuration cell above.
StandardScaler is fitted on each fold's train slice independently.

In [ ]:
X_arr      = X_df.values
y_arr      = y_s.values          # log-scale
timestamps = df.index            # DatetimeIndex — already Timestamps, no wrapping needed

tscv  = TimeSeriesSplit(n_splits=CV_FOLDS, test_size=720)
es_cv = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

fold_results = []

# Naive baseline — log-scale column, back-transform with np.exp() before metrics
naive_raw = X_df[NAIVE_COL].values if NAIVE_COL in feature_cols else None
if naive_raw is None:
    print(f"⚠ Naive col '{NAIVE_COL}' not found — naive metrics will be NaN")

print(f"{'Fold':>4}  {'Month':>10}  {'MAE model':>10}  {'MAE naive':>10}  "
      f"{'RMSE model':>11}  {'RMSE naive':>11}  {'MASE':>6}")
print('-' * 72)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_arr), start=1):

    # Scale
    scaler   = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_arr[train_idx])
    X_te_sc  = scaler.transform(X_arr[test_idx])
    y_tr     = y_arr[train_idx]
    y_te_raw = y_arr[test_idx]

    # Build sequences
    X_tr_seq, y_tr_seq = build_sequences(X_tr_sc, y_tr, INPUT_LENGTH, OUTPUT_LENGTH, HORIZON, STRIDE, shuffle=True)

    lookback_X = X_tr_sc[-INPUT_LENGTH:]
    lookback_y = y_tr[-INPUT_LENGTH:]
    X_te_full  = np.concatenate([lookback_X, X_te_sc])
    y_te_full  = np.concatenate([lookback_y, y_te_raw])
    X_te_seq, y_te_seq = build_sequences(X_te_full, y_te_full, INPUT_LENGTH, OUTPUT_LENGTH, HORIZON, STRIDE, shuffle=False)

    # Train
    model = build_model(n_features, BEST_PARAMS)
    model.fit(
        X_tr_seq, y_tr_seq,
        validation_split=0.15,
        shuffle=False,
        batch_size=BEST_PARAMS['batch_size'],
        epochs=100,
        callbacks=[es_cv],
        verbose=0,
    )

    # Back-transform log-predictions → NZD/MWh before ALL metrics and plots
    y_model  = np.exp(model.predict(X_te_seq, verbose=0).flatten())
    y_actual = np.exp(y_te_seq.flatten())

    # Naive baseline — log-scale column → back-transform
    if naive_raw is not None:
        seq_starts  = np.arange(0, len(X_te_seq)) * STRIDE
        pred_starts = seq_starts + INPUT_LENGTH + HORIZON - 1
        naive_seqs  = np.array([
            naive_raw[test_idx[0] + ps : test_idx[0] + ps + OUTPUT_LENGTH]
            if test_idx[0] + ps + OUTPUT_LENGTH <= len(naive_raw) else np.full(OUTPUT_LENGTH, np.nan)
            for ps in pred_starts
        ])
        y_naive    = np.exp(naive_seqs.flatten())   # back-transform
        valid_mask = ~np.isnan(y_naive)
        mae_n  = mean_absolute_error(y_actual[valid_mask], y_naive[valid_mask])
        rmse_n = np.sqrt(mean_squared_error(y_actual[valid_mask], y_naive[valid_mask]))
    else:
        y_naive = np.full_like(y_actual, np.nan)
        mae_n, rmse_n = np.nan, np.nan

    mae_m  = mean_absolute_error(y_actual, y_model)
    rmse_m = np.sqrt(mean_squared_error(y_actual, y_model))
    mase   = mae_m / mae_n if not np.isnan(mae_n) and mae_n > 0 else np.nan

    month_label = timestamps[test_idx[0]].strftime('%b %Y')

    fold_results.append({
        'fold':       fold,
        'month':      month_label,
        'timestamps': timestamps[test_idx],
        'y_actual':   y_actual,
        'y_model':    y_model,
        'y_naive':    y_naive,
        'mae_model':  mae_m,
        'mae_naive':  mae_n,
        'rmse_model': rmse_m,
        'rmse_naive': rmse_n,
        'mase':       mase,
    })

    print(f"{fold:>4}  {month_label:>10}  {mae_m:>10.2f}  {mae_n:>10.2f}  "
          f"{rmse_m:>11.2f}  {rmse_n:>11.2f}  {mase:>6.3f}")

## Summary Table

In [ ]:
summary = pd.DataFrame([{
    'Month':       r['month'],
    'MAE model':   r['mae_model'],
    'MAE naive':   r['mae_naive'],
    'RMSE model':  r['rmse_model'],
    'RMSE naive':  r['rmse_naive'],
    'MASE':        r['mase'],
    'Beats naive': '✓' if r['mase'] < 1 else '✗'
} for r in fold_results])

avg_row = pd.DataFrame([{
    'Month':       'AVERAGE',
    'MAE model':   summary['MAE model'].mean(),
    'MAE naive':   summary['MAE naive'].mean(),
    'RMSE model':  summary['RMSE model'].mean(),
    'RMSE naive':  summary['RMSE naive'].mean(),
    'MASE':        summary['MASE'].mean(),
    'Beats naive': f"{(summary['MASE'] < 1).sum()} / 12"
}])

pd.set_option('display.float_format', '{:.2f}'.format)
print(pd.concat([summary, avg_row], ignore_index=True).to_string(index=False))

## Monthly Visualisation — Best & Worst Week per Fold

In [ ]:
def find_best_worst_weeks(y_actual, y_model, week_size=168):
    n_weeks = len(y_actual) // week_size
    if n_weeks == 0:
        return (mean_absolute_error(y_actual, y_model), 0, len(y_actual)), \
               (mean_absolute_error(y_actual, y_model), 0, len(y_actual))
    week_maes = []
    for w in range(n_weeks):
        s, e = w * week_size, (w + 1) * week_size
        week_maes.append((mean_absolute_error(y_actual[s:e], y_model[s:e]), s, e))
    return min(week_maes, key=lambda x: x[0]), max(week_maes, key=lambda x: x[0])


for r in fold_results:
    best, worst = find_best_worst_weeks(r['y_actual'], r['y_model'])

    fig, axes = plt.subplots(2, 1, figsize=(16, 9))
    fig.suptitle(
        f"{r['month']}  —  MAE model: {r['mae_model']:.1f}  "
        f"naive: {r['mae_naive']:.1f}  MASE: {r['mase']:.3f}",
        fontsize=12
    )

    for ax, (mae, s, e), label in zip(
        axes,
        [best, worst],
        ['Best week (lowest MAE)', 'Worst week (highest MAE)']
    ):
        act = r['y_actual'][s:e]
        mod = r['y_model'][s:e]
        nav = r['y_naive'][s:e]

        ax.plot(act, label='Actual',  color='steelblue',  linewidth=1.5)
        ax.plot(mod, label='LSTM',    color='darkorange', linewidth=1.5, linestyle='--')
        if not np.isnan(nav).all():
            ax.plot(nav, label='Naive', color='gray',      linewidth=1.0, linestyle=':')
        ax.set_title(f"{label}  —  MAE: {mae:.1f} NZD/MWh")
        ax.set_ylabel('Price (NZD/MWh)')
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=8, loc='upper right')
        ax.tick_params(axis='x', labelrotation=30)

    plt.tight_layout()
    plt.show()

## Final Model — trained on 80% with best params

In [ ]:
# Scale on full training set
final_scaler  = StandardScaler()
X_tr_final_sc = final_scaler.fit_transform(X_train_df.values)
X_te_final_sc = final_scaler.transform(X_test_df.values)

X_tr_fin_seq, y_tr_fin_seq = build_sequences(
    X_tr_final_sc, y_train_s.values,
    INPUT_LENGTH, OUTPUT_LENGTH, HORIZON, STRIDE, shuffle=True
)

lookback_X_fin = X_tr_final_sc[-INPUT_LENGTH:]
lookback_y_fin = y_train_s.values[-INPUT_LENGTH:]
X_te_fin_full  = np.concatenate([lookback_X_fin, X_te_final_sc])
y_te_fin_full  = np.concatenate([lookback_y_fin, y_test_s.values])
X_te_fin_seq, y_te_fin_seq = build_sequences(
    X_te_fin_full, y_te_fin_full,
    INPUT_LENGTH, OUTPUT_LENGTH, HORIZON, STRIDE, shuffle=False
)

es_final = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

final_model = build_model(n_features, BEST_PARAMS)
final_model.fit(
    X_tr_fin_seq, y_tr_fin_seq,
    validation_split=0.15,
    shuffle=False,
    batch_size=BEST_PARAMS['batch_size'],
    epochs=150,
    callbacks=[es_final],
    verbose=1,
)

# Back-transform log-predictions → NZD/MWh before metrics
y_pred_final = np.exp(final_model.predict(X_te_fin_seq, verbose=0).flatten())
y_true_final = np.exp(y_te_fin_seq.flatten())

mae_f  = mean_absolute_error(y_true_final, y_pred_final)
rmse_f = np.sqrt(mean_squared_error(y_true_final, y_pred_final))

print('Final model — best params from Optuna search')
print(f'  MAE  : {mae_f:.4f} NZD/MWh')
print(f'  RMSE : {rmse_f:.4f} NZD/MWh')
print(f'\nBest params used:')
for k, v in sorted(BEST_PARAMS.items()):
    print(f'  {k:<18}: {v}')

## Save Final Model

In [ ]:
import os
from datetime import datetime

# Build a descriptive filename so the saved model is self-documenting
date_str    = datetime.today().strftime('%Y%m%d')
window_str  = f"{DATE_START.replace('-','') if DATE_START else 'full'}_{DATE_END.replace('-','') if DATE_END else 'full'}"
params_str  = (f"L1-{BEST_PARAMS['lstm1_units']}"
               f"_L2-{BEST_PARAMS['lstm2_units']}"
               f"_do-{BEST_PARAMS['dropout']}"
               f"_lr-{BEST_PARAMS['learning_rate']}"
               f"_bs-{BEST_PARAMS['batch_size']}")
model_name  = f"LSTM_OTA2201_{date_str}_{window_str}_in{INPUT_LENGTH}_{params_str}_MAE{mae_f:.1f}.keras"

os.makedirs('saved_models', exist_ok=True)
save_path = os.path.join('saved_models', model_name)
final_model.save(save_path)

print(f"Model saved to: {save_path}")